# Section1 Mongolian News Classification

This notebook contains a simple demo classifying the [Eduge news dataset](https://github.com/tugstugi/mongolian-nlp) provided by [Bolorsoft LLC](https://bolorsoft.com/) using a SVM and [SentencePiece](https://github.com/google/sentencepiece).

## Download Eduge dataset

[notebook from](https://github.com/sharavsambuu/mongolian-text-classification) sharavsambuu

In [1]:
import os
from os.path import exists, join, basename, splitext

if not exists("eduge.csv"):
  !wget -q https://github.com/tugstugi/mongolian-nlp/raw/master/datasets/eduge.csv.gz
  !gunzip eduge.csv.gz

## Download SentencePiece vocabulary

A SentencePiece model trained on a Mongolian corpus containg 650M words will be used the text tokenizer. We will download it from the repo [tugstugi/mongolian-bert](https://github.com/tugstugi/mongolian-bert):


In [2]:
if not exists('mn_uncased.model'):
  # download both SentencePiece models: cased and uncased
  !wget -q https://github.com/tugstugi/mongolian-bert/raw/master/sentencepiece/mn_cased.model
  !wget -q https://github.com/tugstugi/mongolian-bert/raw/master/sentencepiece/mn_cased.vocab
  !wget -q https://github.com/tugstugi/mongolian-bert/raw/master/sentencepiece/mn_uncased.model
  !wget -q https://github.com/tugstugi/mongolian-bert/raw/master/sentencepiece/mn_uncased.vocab
    
  # install SentencePiece
  !pip install -q sentencepiece

## Load SentencePiece and test

Load the downloaded SentencePiece model and tokenize some text:

In [3]:
import sentencepiece as spm
import pandas as pd
import numpy as np
import time

from sklearn.feature_extraction.text import *
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

sp = spm.SentencePieceProcessor()
sp.Load('mn_cased.model')
def sp_tokenize(w):
  return sp.EncodeAsPieces(w)

" ".join(sp_tokenize('Мөнгөө тушаачихсаныхаа дараа мэдэгдээрэй'.lower()))

'▁мөнгөө ▁тушаа чихсан ыхаа ▁дараа ▁мэдэгд ээрэй'

## Train/Test split

In [4]:
df = pd.read_csv("eduge.csv")
df = df.rename(columns=lambda x: x.strip())
labels = df['label'].unique().tolist()
# show labels
print('labels', df['label'].unique().tolist())

# stratified train and test split
train, test = train_test_split(df, test_size=0.1, random_state=999, stratify=df['label'])

labels ['урлаг соёл', 'эдийн засаг', 'эрүүл мэнд', 'хууль', 'улс төр', 'спорт', 'технологи', 'боловсрол', 'байгал орчин']


## Train SVM

Now train a SVM, no hyperparameter optimization, use only some default parameters:

In [5]:
text_clf = Pipeline([ ('vect', CountVectorizer(tokenizer=sp_tokenize, lowercase=True)),
                        # ('vect', TfidfVectorizer(tokenizer=sp_tokenize)),

                         ('tfidf', TfidfTransformer()),

                        #  ('clf', AdaBoostClassifier()) ])
                        # ('clf', RandomForestClassifier()) ])
                         ('clf', SGDClassifier(loss='hinge', penalty='l2', alpha=1e-4, random_state=0))])

t = time.time()
text_clf = text_clf.fit(train['news'], train['label'])
t = time.time()-t
print("Training time in seconds: ", t)

t = time.time()
predicted = text_clf.predict(test['news'])
t = time.time()-t
print("Prediction time in seconds: ", t)


print("Feature count:", len(text_clf.named_steps['vect'].vocabulary_))
print("Classifier accuracy: ", np.mean(predicted == test['label']))

Training time in seconds:  52.022350788116455
Prediction time in seconds:  6.228482246398926
Feature count: 25384
Classifier accuracy:  0.91053257565746


### News from GoGo.mn

In [6]:
ex_text_str1 = "FIBA-ын 2025 оны Азийн аварга шалгаруулах тэмцээний урьдчилсан нэгдүгээр шатны D хэсгийн тоглолтууд Улаанбаатар хотод болж өндөрлөлөө. Таван өдөр үргэлжилсэн хэсгийн тоглолтууд дуусаж хэсгийн тэргүүн багууд тодров. Өнөөдөр Монголын шигшээ баг болон Тайландын шигшээ багууд хэсгийн аваргын төлөө тоглоход Тайландын шигшээ баг 109-105 харьцаагаар хожлоо. Монголын шигшээ баг тоглолтын эхний хагаст 10 онооны зөрүүгээр тэргүүлж байсан ч гуравдугаар үед Тайланд илүү тоглож оноо барьсан. Улмаар дөрөвдүгээр үед Тайланд тэргүүллээ баталгаажуулж багийн гол тоглогч Фредерих Лишийн дараалсан холын шидэлтүүд байгаа оносноор Монголчууд урьдчилсан шатанд анх удаа хожигдлоо. Энэ тоглолтын өмнө Вьетнам, Малайзын шигшээ багууд хэн нь хэсгийн гуравдугаар байраар дараагийн шатанд шалгарахаа шийдэж тоглоход Малайз 73-56 харьцаагаар хожлоо. Урьдчилсан нэгдүгээр шатны С хэсгээс Гуам 4, Хонконг 3, Сингапур 2 оноотой хоёрдугаар шатанд шалгарсан бол D хэсгээс ийнхүү Тайланд, Монгол, Малайз шалгарав. Энэ зургаан баг урьдчилсан хоёрдугаар шатанд 2023 оны хоёрдугаар сард тоглоно. Үүнээс хамгийн өндөр оноо авсан 4 баг нь Азийн аваргын урьдчилсан гуравдугаар шатанд шалгарч тоглох учиртай."

In [7]:
predicted = text_clf.predict([ex_text_str1])
print(predicted)

['спорт']


In [8]:
ex_text_str2 = "Samsung Experience Store-ийн тохижилт, үйлчилгээний стандартыг Samsung брэндийн бодлогын дагуу хийж гүйцэтгэсэн байна. Sain electronics компани нь Samsung Experience store-оор дамжуулан Samsung брэндийн Албан ёсны гар утас, телевиз, тоос сорогч, цахилгаан бараа болон дагалдах хэрэгслийг хэрэглэгчдэдээ нэг дороос үзэж, сонгон, худалдан авалт хийх боломжоор хангаж байгаа анхны компани юм"

In [9]:
predicted = text_clf.predict([ex_text_str2])
print(predicted)

['технологи']


In [10]:
ex_text_str3 = "FIBA-ын 2025 оны Азийн аварга шалгаруулах тэмцээний урьдчилсан нэгдүгээр шатны D хэсгийн тоглолтууд Улаанбаатар хотод дөрөв дэх өдрөө үргэлжилж байна. Монголын шигшээ баг өнөөдөр гурав дахь тоглолтоо Вьетнам улсын шигшээ багтай хийсэн бөгөөд 88-77 харьцаагаар хожлоо. Шигшээ баг уржигдар Малайзыг хүндхэн тоглолтын эцэст 94-92 харьцаагаар, өчигдөр Таитиг 62-91 харьцаагаар илүүрхэн хожоод байв. Ийнхүү өнөөдөр хэсэгтээ гурван хожилтой болж дараагийн шатанд шалгаран тоглох болзлоо хангалаа. Тэд маргааш хэсгийг тэргүүлэгч Тайландын шигшээ багтай тоглож хэсгийн хэддүгээр байраар шалгарахаа шийднэ."

In [11]:
predicted = text_clf.predict([ex_text_str3])
print(predicted)

['спорт']


## Problem1: Test and classify any news from WEB
## Problem: Increase text classification model accuracy rate [Optional]

# Section2 Simple Plagiarism Detection Using NLP

The birth of the World Wide Web saw an increased sharing of knowledge and resources all across the world. However, this has also resulted in plagiarism across different domains. Academic literature is highly prone to plagiarism, where ideas and work are stolen and replicated without proper acknowledgment.

Natural Language Processing technologies can be used to effectively to detect plagiarism in texts. Here, we see NLP applied to detect external plagiarism, i.e when both the original text as well as the suspicious text are available to us.

Before using other NLP techniques, we first apply pre-processing techniques to the text. This improves the accuracy of the plagiarism detection approach used.

In [12]:
pipe = Pipeline([('count', CountVectorizer(tokenizer=sp_tokenize, lowercase=True)),
                 ('tfid', TfidfTransformer())])
pipe.fit(train)

Pipeline(steps=[('count',
                 CountVectorizer(tokenizer=<function sp_tokenize at 0x7ff5fc30be20>)),
                ('tfid', TfidfTransformer())])

In [13]:
vecs = pipe.transform([ex_text_str1, ex_text_str2, ex_text_str3]).toarray()

In [14]:
print(vecs)

[[0.         0.         0.         1.         0.        ]
 [0.         0.14142136 0.         0.98994949 0.        ]
 [0.         0.         0.         1.         0.        ]]


In [15]:
from sklearn.metrics.pairwise import cosine_distances
dist1 = cosine_distances([vecs[0]],[vecs[1]])
dist2 = cosine_distances([vecs[1]],[vecs[2]])
dist3 = cosine_distances([vecs[0]],[vecs[2]])
print(dist1, dist2, dist3)

[[0.01005051]] [[0.01005051]] [[0.]]


## Problem2: Implement your "Own Plagiarism" detection model.

# Section3 Mongolian to English Translation [Optional]

Check [colab Pretrained version for fast inference](https://github.com/sharavsambuu/english-mongolian-nmt-dataset-augmentation)

Idea 1; change Mongolian-English to Mongolian-Japanese-English, which may appear better results. Because Japanese-English translation implemented well and Mongolian and Japanese have same [language family](https://en.wikipedia.org/wiki/Altaic_languages), which easier to translate Mongolian to Japanese.

Idea 2; train Mongolian-English and English-Mongolian models, cycle translation model annotation.

https://pypi.org/project/googletrans/ no longer working!!!

In [21]:
!ls

Eduge_SVM_baseline.ipynb  mn_cased.model  mn_uncased.model  tokenizer_en.pickle
checkpoints		  mn_cased.vocab  mn_uncased.vocab  tokenizer_mn.pickle
eduge.csv		  mn_en.txt	  precomputed.zip


In [22]:
import time
import numpy as np
from sklearn.model_selection import train_test_split
import codecs


In [23]:
examples = []
file_name = "mn_en.txt"
with codecs.open(file_name, 'r', encoding='utf-8', errors='ignore') as f:
  while True:
    line = f.readline()
    if not line:
      break
    arr = line.split("+++++SEP+++++")
    if len(arr)==2:
      examples.append((arr[0], arr[1]))

In [24]:
train_examples, val_examples = train_test_split(examples, test_size=0.2)

In [27]:
print(train_examples[0][0])
print(train_examples[0][1])

Бидний зөвлөх бол улс төрийн салбарт өндөр авчихсан хүн, зөвлөгөө нь их тустай авууштай. 
 Our advisors are very helpful in politics and advice.



In [28]:
Google_translate = "Our consultant is a highly experienced person in the political field, and his advice is very helpful."

In [ ]:
imtranslator = "Our advisor is a highly helpful man and advice has been taken high in political fields."
imtranslator_back = "Манай зөвлөх бол өндөр тустай хүн бөгөөд улс төрийн чиглэлээр өндөр зөвлөгөө өгдөг."